2.卷积和池化层


![2.1理论计算题](2.1理论计算题.png)

In [ ]:
import torch

def max_pool2d_forward(x, kernel_size, stride, padding):
    """
    :param x: 输入张量 [N,C,H,W]
    :param kernel_size: int/(kh,kw)
    :param stride: int/(sh,sw)
    :param padding: int/(ph,pw)
    :return: pool输出 [N,C,Ho,Wo]
    """
    # 统一参数为二元组
    if isinstance(kernel_size, int):
        kh, kw = kernel_size, kernel_size
    else:
        kh, kw = kernel_size
    if isinstance(stride, int):
        sh, sw = stride, stride
    else:
        sh, sw = stride
    if isinstance(padding, int):
        ph, pw = padding, padding
    else:
        ph, pw = padding

    N, C, H_in, W_in = x.shape
    # 四边填充：左、右、上、下
    x_pad = torch.nn.functional.pad(x, (pw, pw, ph, ph), value=0.)
    H_pad, W_pad = x_pad.shape[2], x_pad.shape[3]

    # 输出尺寸计算公式
    Ho = (H_pad - kh) // sh + 1
    Wo = (W_pad - kw) // sw + 1
    out = torch.zeros((N, C, Ho, Wo))

    # 四重循环逐窗口取最大值
    for n in range(N):
        for c in range(C):
            for i in range(Ho):
                h_start = i * sh
                h_end = h_start + kh
                for j in range(Wo):
                    w_start = j * sw
                    w_end = w_start + kw
                    window = x_pad[n, c, h_start:h_end, w_start:w_end]
                    out[n, c, i, j] = torch.max(window)
    return out

# 测试验证
if __name__ == "__main__":
    x = torch.randn(2,3,8,8)
    # 自定义池化
    res_custom = max_pool2d_forward(x, kernel_size=2, stride=2, padding=0)
    # 官方API对照
    pool_official = torch.nn.MaxPool2d(kernel_size=2, stride=2, padding=0)
    res_official = pool_official(x)
    print("输出形状：", res_custom.shape)
    print("与官方结果一致：", torch.allclose(res_custom, res_official))

输出形状： torch.Size([2, 3, 4, 4])
与官方结果一致： True


3.LeNet, AlexNet, VGG 和NiN

![3.1理论计算题](3.1理论计算题.png)

In [2]:
import torch
import torch.nn as nn

class NiNBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride, padding):
        super().__init__()
        # 结构：普通卷积+ReLU → 1×1卷积+ReLU → 1×1卷积+ReLU
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=kernel_size, stride=stride, padding=padding),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=1, stride=1, padding=0),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=1, stride=1, padding=0),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.block(x)

# 测试示例
if __name__ == "__main__":
    # 输入通道3，输出通道16，3×3卷积、步幅1、填充1
    nin = NiNBlock(in_channels=3, out_channels=16, kernel_size=3, stride=1, padding=1)
    test_x = torch.randn(2, 3, 32, 32)
    out = nin(test_x)
    print("输出shape:", out.shape)

输出shape: torch.Size([2, 16, 32, 32])


4.Inception, 批量归一化和残差网络


![4.1理论计算题](4.1理论计算题.png)

In [4]:
import torch
import torch.nn as nn

class Residual(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, use_1x1conv=False):
        super().__init__()
        # 两个3×3卷积+BN，输出通道一致
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)

        # use_1x1conv=True时，捷径用1×1卷积匹配维度
        self.shortcut = nn.Sequential()
        if use_1x1conv:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        out = torch.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        # 残差相加 f(x)+x(shortcut映射后)
        out = out + self.shortcut(x)
        out = torch.relu(out)
        return out

# 测试
if __name__ == "__main__":
    # 1、不使用1×1卷积，通道尺寸相同
    res1 = Residual(3,3,stride=1,use_1x1conv=False)
    x1 = torch.rand(2,3,32,32)
    print(res1(x1).shape)

    # 2、使用1×1卷积，通道/尺寸变化
    res2 = Residual(3,16,stride=2,use_1x1conv=True)
    x2 = torch.rand(2,3,32,32)
    print(res2(x2).shape)

torch.Size([2, 3, 32, 32])
torch.Size([2, 16, 16, 16])


5.图像增广，微调和样式迁移

5.1理论计算题

1、底层小学习率、顶层大学习率原因
底层预训练特征通用：在 ImageNet 预训练的底层卷积提取边缘、纹理、基础轮廓等通用视觉特征，参数已经收敛到优质最优解，不需要大幅改动；设置小lr/冻结可以避免在小目标数据集上破坏通用特征，防止原有有效权重被错误更新。
顶层分类全新适配：输出层是随机初始化，没有经过训练，需要较大学习率快速拟合目标数据集的类别分布，快速收敛适配新任务。
数据分布差异：源域与目标域分布存在差距，顶层需要快速适配新标签，底层微调幅度越小越不容易过拟合。

2、小样本+同源数据集防过拟合微调方案
冻结全部特征提取层：固定主干网络所有卷积、BN 参数，只训练最后全新的分类输出层，主干完全复用预训练特征，参数量最小化，最大限度抑制过拟合。
加入正则手段：训练时搭配 Dropout、权重衰减L2正则、早停 (Early Stop)；同时使用图像增广扩充有限样本。
超参控制：使用很小的学习率，减小迭代轮数，避免在少量样本上过拟合。

In [1]:
from torchvision import transforms
from PIL import Image

train_transform = transforms.Compose([
    # 1.随机裁剪：面积0.08~1.0，缩放至224×224
    transforms.RandomResizedCrop(size=224, scale=(0.08, 1.0)),
    # 2.50%概率水平翻转
    transforms.RandomHorizontalFlip(p=0.5),
    # 3.随机亮度、对比度、饱和度变化0.5
    transforms.ColorJitter(brightness=0.5, contrast=0.5, saturation=0.5, hue=0.),
    # 4.转为Tensor
    transforms.ToTensor()
])

img = Image.open("test.jpg") # 替换成你本地图片路径
img_tensor = train_transform(img)
print("变换后张量shape：", img_tensor.shape)

变换后张量shape： torch.Size([3, 224, 224])


6.目标检测，计算机视觉训练技巧

![6.1理论计算题](6.1理论计算题.png)

In [2]:
import torch
import torch.nn.functional as F

def label_smooth_cross_entropy(logits, target, eps=0.1):
    """
    logits: [N, K] 模型原始输出（未经过softmax）
    target: [N] 真实类别索引
    eps: 平滑系数，题目默认0.1
    K: 分类总数
    """
    N, K = logits.shape
    # 1、softmax得到预测概率
    pred = F.softmax(logits, dim=-1)
    log_pred = torch.log(pred + 1e-8)

    # 2、构造平滑标签
    smooth_label = torch.full_like(log_pred, fill_value=eps/(K-1))
    # 正确类别赋值 1-eps
    smooth_label.scatter_(dim=1, index=target.unsqueeze(1), value=1. - eps)

    # 3、交叉熵：-∑ y_smooth * log(p)
    loss = -torch.sum(smooth_label * log_pred, dim=-1)
    return torch.mean(loss)

# 测试示例
if __name__ == "__main__":
    # batch=4，5分类
    logits = torch.randn(4,5)
    label = torch.tensor([0,2,1,3])
    loss = label_smooth_cross_entropy(logits, label, eps=0.1)
    print("标签平滑损失值：", loss.item())

标签平滑损失值： 2.1069083213806152
